In [1]:
from toy_model_copy import *
from metrics import *
import wandb
import torch
import numpy as np

/opt/anaconda3/envs/toytrans/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

wandb.login()

wandb: Currently logged in as: ce24b119 (jerrycloud3316-ai-club-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# to find kl loss between model and these processes
T0_proc1 = np.array([
    [0, 1, 0],
    [0, 0, 1], 
    [0, 0, 0.5]
])
T1_proc1 = np.array([
    [0, 0, 0],
    [0, 0, 0],
    [0.5, 0, 0]
])

# Different process
T0_proc2 = np.array([
    [0,1,0],
    [0,0,0],
    [0.5,0,0]
])
T1_proc2 = np.array([
    [0,0,0],
    [0,0,1],
    [0.5,0,0]
])
T0_proc3 = np.array([
    [0,1,0,0],
    [0,0,0,0.5],
    [0.5,0,0,0],
    [0,0,0,0.5]])
T1_proc3 = np.array([
    [0,0,0,0],
    [0,0,0.5,0],
    [0.5,0,0,0],
    [0.5,0,0,0]])   
process1 = MarkovData(n_gen=100, gen_len=31, n_states=3, d_vocab=2, T_list=[T0_proc1, T1_proc1], seed=43,prepend_bos=True)
process2 = MarkovData(n_gen=100, gen_len=31, n_states=3, d_vocab=2, T_list=[T0_proc2, T1_proc2], seed=43,prepend_bos=True)
process3 = MarkovData(n_gen=100, gen_len=31, n_states=4, d_vocab=2, T_list=[T0_proc3, T1_proc3], seed=43,prepend_bos=True)

In [4]:
dataset=MarkovData(n_gen=10000, gen_len=31, n_states=3, d_vocab=2, T_list=[T0_proc1, T1_proc1], seed=43,prepend_bos=True)

In [11]:
dataset1=MarkovData(n_gen=5000, gen_len=31, n_states=3, d_vocab=2, T_list=[T0_proc1, T1_proc1],prepend_bos=True)
dataset2=MarkovData(n_gen=5000, gen_len=31, n_states=3, d_vocab=2, T_list=[T0_proc2, T1_proc2],prepend_bos=True)
merged_dataset = MergeMarkovDatasets(dataset1=dataset1, dataset2=dataset2,mixing_style='random')

In [12]:
print(dataset[0])
print(process1[0])
for i in range(10):
    print(merged_dataset[i])

{'tokens': tensor([2, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 1, 0, 0])}
{'tokens': tensor([2, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 1, 0, 0])}
{'tokens': tensor([2, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1,
        0, 0, 1, 0, 0, 1, 0, 0])}
{'tokens': tensor([2, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 1, 0, 0])}
{'tokens': tensor([2, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 1, 0])}
{'tokens': tensor([2, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 1, 0, 0, 1, 0, 0])}
{'tokens': tensor([2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0])}
{'tokens': tensor([2, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 1, 0, 1, 0, 0, 1, 1])}


In [13]:

metrics_config = MetricsConfig(
    track_markov_kl=True,
    markov_processes=[process1,process2,process3], 
    pos_start=6,
    
    track_ngrams=False,
    ngram_orders=[1, 2, 3],
    track_previous_token=False,
    track_in_context=False, 
    icl_k1=5,
    icl_k2=32,
    track_composition=False,
    track_prefix_matching=False)

In [14]:

model = train_model(
    dataset=merged_dataset,
    n_layers=2,
    d_model=16,
    n_heads=2, 
    attn_only=False,
    act_fn='silu',
    normalization_type='LN',

    n_epochs=300,
    batch_size=64,
    lr=0.05,

    wandb=True,
    wandb_project_name="ICL",
    save_dir="proc1/seq_len32/X7*_ln_bos",
    save_every=20,
    print_every=10,


    metrics_config=metrics_config,
    metrics_log_interval=20
    )
wandb.finish()

Moving model to device:  cpu


  0%|          | 0/300 [00:00<?, ?it/s]

Metrics logged at step 50
Metrics logged at step 100


  0%|          | 1/300 [00:02<10:39,  2.14s/it]

Epoch 1 Validation Loss 0.6212044954299927
Metrics logged at step 150
Metrics logged at step 200


...

100%|█████████▉| 299/300 [11:04<00:02,  2.15s/it]

Epoch 299 Validation Loss 0.33427149057388306
Metrics logged at step 37400
Metrics logged at step 37450


100%|██████████| 300/300 [11:06<00:00,  2.22s/it]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Metrics logged at step 37500
Epoch 300 Samples 8000 Step 124 Training Loss 0.32842519879341125
Epoch 300 Validation Loss 0.33432286977767944


epoch,▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇█████
markov0_to_model_kl,▁▂▂▂▃▃▄▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇██████████████████
markov1_to_model_kl,▁▁▁▂▂▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇██████
markov2_to_model_kl,▁▂▂▃▄▅▆▅▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████
model_to_markov0_kl,▁▅▆▇▇▆█▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇█▇█▇▇▇▇▇▇▇▇▇
model_to_markov1_kl,██▄▂▂▂▁▂▂▂▂▂▂▁▁▂▁▂▁▂▂▂▂▂▁▂▁▁▁▂▁▂▂▁▁▂▁▂▂▂
model_to_markov2_kl,▃▄▁▅▅▇█▆▆▆▇▇▆▆▆▇▆▆▇▆▆▆▇▇▇▆▆▆▇▇▇▇▆▆▇▇▇▇▇▆
samples,▆█▂▃▆▇▃▂▃▄▄▁▇▄▆▂▃▄▄▅▃▄▅▅▂▄▄▂▂▅▅█▄▂▁▆▅█▆▁
train_loss,█▃▃▂▂▂▂▁▂▁▁▁▁▁▂▁▁▁▁▁▂▂▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▂▁
val_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,300
